# BirdCLEF+


In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import timm
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)
print('Libraries loaded ✓')

## Config

In [ ]:
class CFG:
    # Audio
    SR          = 32000          # sample rate
    DURATION    = 5              # seconds per clip
    N_MELS      = 128            # mel bins
    N_FFT       = 1024
    HOP_LENGTH  = 320            # → ~100 frames/sec
    FMIN        = 50
    FMAX        = 14000

    # Training
    BATCH_SIZE  = 32
    EPOCHS      = 15
    LR          = 1e-3
    VAL_FRAC    = 0.1            
    NUM_WORKERS = 2

    # Paths  (Kaggle layout)
    TRAIN_CSV   = '/kaggle/input/birdclef-2026/train.csv'
    AUDIO_DIR   = '/kaggle/input/birdclef-2026/train_audio/'
    MODEL_OUT   = 'baseline_model_best.pth'

    # Model
    BACKBONE    = 'efficientnet_b0'
    PRETRAINED  = True

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {CFG.DEVICE}')

## Dataset

In [ ]:
class BirdDataset(Dataset):
    """
    Loads a fixed-length audio clip, converts it to a log-mel spectrogram,
    and returns (spectrogram_tensor, multi-hot_label_tensor).
    """
    def __init__(self, df: pd.DataFrame, audio_dir: str, mlb: MultiLabelBinarizer, augment: bool = False):
        self.df        = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.mlb       = mlb
        self.augment   = augment

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path: str) -> np.ndarray:
        """Load audio, trim/pad to exactly DURATION seconds."""
        y, _ = librosa.load(path, sr=CFG.SR, mono=True)
        target_len = CFG.SR * CFG.DURATION

        if len(y) < target_len:
            reps = int(np.ceil(target_len / len(y)))
            y    = np.tile(y, reps)[:target_len]
        else:
            if self.augment:
                start = np.random.randint(0, len(y) - target_len)
            else:
                start = (len(y) - target_len) // 2
            y = y[start:start + target_len]
        return y

    def _to_melspec(self, y: np.ndarray) -> np.ndarray:
        """Convert waveform → normalised log-mel spectrogram (1, H, W)."""
        mel = librosa.feature.melspectrogram(
            y=y, sr=CFG.SR,
            n_mels=CFG.N_MELS, n_fft=CFG.N_FFT,
            hop_length=CFG.HOP_LENGTH,
            fmin=CFG.FMIN, fmax=CFG.FMAX
        )
        mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return mel[np.newaxis]  # (1, N_MELS, T)

    def _augment_spec(self, mel: np.ndarray) -> np.ndarray:
        """Simple SpecAugment: random frequency & time masking."""
        mel = mel.copy()
        _, H, W = mel.shape
        # Frequency mask
        f = np.random.randint(0, H // 8)
        f0 = np.random.randint(0, H - f)
        mel[:, f0:f0+f, :] = 0
        # Time mask
        t = np.random.randint(0, W // 8)
        t0 = np.random.randint(0, W - t)
        mel[:, :, t0:t0+t] = 0
        return mel

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['filename'])

        y   = self._load_audio(path)
        mel = self._to_melspec(y)

        if self.augment:
            mel = self._augment_spec(mel)

        # Labels: row['labels'] is already a list after preprocessing
        label = self.mlb.transform([row['labels']])[0].astype(np.float32)

        return torch.from_numpy(mel), torch.from_numpy(label)

## Model — EfficientNet-B0 + sigmoid head

In [ ]:
class BirdModel(nn.Module):
    """
    EfficientNet-B0 backbone accepting single-channel mel spectrograms.
    Outputs raw logits (apply sigmoid for probabilities).
    """
    def __init__(self, num_classes: int):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.BACKBONE,
            pretrained=CFG.PRETRAINED,
            in_chans=1,         
            num_classes=0,       
            global_pool='avg'
        )
        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats  = self.backbone(x)   
        logits = self.head(feats)   
        return logits

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        """Returns sigmoid probabilities — use this at inference time."""
        return torch.sigmoid(self.forward(x))

## Training & validation loops

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(CFG.DEVICE), y.to(CFG.DEVICE)
        optimizer.zero_grad()

        with autocast():                   
            logits = model(x)
            loss   = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(CFG.DEVICE), y.to(CFG.DEVICE)

        with autocast():
            logits = model(x)
            loss   = criterion(logits, y)

        total_loss   += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(y.cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    valid_cols = all_labels.sum(axis=0) > 0
    if valid_cols.sum() > 0:
        auc = roc_auc_score(all_labels[:, valid_cols], all_preds[:, valid_cols],
                            average='macro')
    else:
        auc = 0.0

    return total_loss / len(loader), auc

## data prep + training loop

In [ ]:
def main():
    df = pd.read_csv(CFG.TRAIN_CSV)
    print(f'Total samples: {len(df)}')

    if df['labels'].dtype == object:
        df['labels'] = df['labels'].apply(
            lambda x: x.split() if isinstance(x, str) else [x]
        )

    mlb = MultiLabelBinarizer()
    mlb.fit(df['labels'])
    num_classes = len(mlb.classes_)
    print(f'Number of classes: {num_classes}')

    train_df, val_df = train_test_split(
        df, test_size=CFG.VAL_FRAC, random_state=42, shuffle=True
    )
    print(f'Train: {len(train_df)} | Val: {len(val_df)}')

    train_ds = BirdDataset(train_df, CFG.AUDIO_DIR, mlb, augment=True)
    val_ds   = BirdDataset(val_df,   CFG.AUDIO_DIR, mlb, augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=CFG.BATCH_SIZE,
        shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.BATCH_SIZE,
        shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True
    )

    model     = BirdModel(num_classes=num_classes).to(CFG.DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.EPOCHS, eta_min=1e-6
    )
    criterion = nn.BCEWithLogitsLoss()   
    scaler    = GradScaler()             

    best_auc = 0.0

    for epoch in range(1, CFG.EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_loss, val_auc = validate(model, val_loader, criterion)
        scheduler.step()

        print(f'Epoch {epoch:02d}/{CFG.EPOCHS}  '
              f'train_loss={train_loss:.4f}  '
              f'val_loss={val_loss:.4f}  '
              f'val_AUC={val_auc:.4f}')

        if val_auc >= best_auc:
            best_auc = val_auc
            torch.save({
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'val_auc':     val_auc,
                'classes':     list(mlb.classes_)
            }, CFG.MODEL_OUT)
            print(f'  ↑ Best model saved  (AUC={best_auc:.4f})')

    print(f'\nTraining complete. Best val AUC: {best_auc:.4f}')
    print(f'Checkpoint saved to: {CFG.MODEL_OUT}')
    return mlb


mlb = main()

## output prediction probabilities

In [ ]:
@torch.no_grad()
def predict_from_file(audio_path: str, model: BirdModel) -> pd.Series:


    dummy_ds = BirdDataset.__new__(BirdDataset)
    dummy_ds.audio_dir = ''
    dummy_ds.mlb       = mlb
    dummy_ds.augment   = False

    y   = dummy_ds._load_audio(audio_path)
    mel = dummy_ds._to_melspec(y)
    x   = torch.from_numpy(mel).unsqueeze(0).to(CFG.DEVICE)   # (1, 1, H, W)

    model.eval()
    probs = model.predict_proba(x).cpu().numpy()[0]            # (num_classes,)

    return pd.Series(probs, index=mlb.classes_).sort_values(ascending=False)

ckpt = torch.load(CFG.MODEL_OUT, map_location=CFG.DEVICE)
num_classes = len(ckpt['classes'])

loaded_model = BirdModel(num_classes=num_classes).to(CFG.DEVICE)
loaded_model.load_state_dict(ckpt['model_state'])
print(f'Loaded checkpoint from epoch {ckpt["epoch"]} (val AUC={ckpt["val_auc"]:.4f})')
print(f'Model outputs probabilities for {num_classes} species.')
